# Final Capstone: Stock Market Reaction Justification

---
## Importing Requirements

In [17]:
import pandas as pd
import numpy
import matplotlib
import seaborn
import sqlalchemy
import yfinance as yf
from fredapi import Fred

---
## Data

In [146]:
# Fred api key
fred = Fred(api_key="7f0fb7b341f36af702bff6dba49512d3")

# tables of macroeconomic data from Fred
cpi = fred.get_series("cpiaucsl")
interest = fred.get_series("fedfunds")
unemployment = fred.get_series("unrate")
market_expect = fred.get_series("dgs10")
volatility = fred.get_series("vixcls")

# stock data from yfinance
sp500 = yf.download("^GSPC", period="max")
sp500.head()
sp500.tail()
sp500.shape
sp500.index.min()
sp500.index.max()

# convert data to a dataframe 
cpi = cpi.to_frame(name="cpi")
interest = interest.to_frame(name="fedfunds")
unemployment = unemployment.to_frame(name="unrate")
market_expect = market_expect.to_frame(name="dgs10")
volatility = volatility.to_frame(name="vix")

# merge macroeconomic dataframes
macro_df = cpi.join([interest, unemployment, market_expect, volatility], how="outer")
macro_df.sort_index()

# convert variables into meansuring changes
macro_df["cpi_change"] = macro_df["cpi"].pct_change()
macro_df["fedfunds_change"] = macro_df["fedfunds"].diff()
macro_df["unrate_change"] = macro_df["unrate"].diff()
macro_df["mrkt_expect_change"] = macro_df["dgs10"].diff()
macro_df["volatility_change"] = macro_df["vix"].pct_change()
macro_df.sort_index()

# clean stock data
sp500.columns = sp500.columns.get_level_values(0)
sp500 = sp500[["Close"]]
sp500 = sp500.rename(columns={"Close": "sp500"})
sp500["return"] = sp500["sp500"].pct_change()
sp500 = sp500.reset_index()
sp500 = sp500.rename(columns={"Date": "date"})
sp500["date"] = pd.to_datetime(sp500["date"])
sp500

# make macro and stock data monthly
macro_df = macro_df.reset_index()
macro_df = macro_df.rename(columns={"index": "date"})
macro_df["date"] = pd.to_datetime(macro_df["date"])
macro_monthly = macro_df.set_index("date").resample("ME").last()
sp500_monthly = sp500.set_index("date").resample("ME").last()
sp500_monthly

# merge macro_daily and sp500
sp500_macro = pd.merge(sp500_monthly, macro_monthly, on="date", how="outer")
sp500_macro


[*********************100%***********************]  1 of 1 completed


,sp500,return,cpi,fedfunds,unrate,dgs10,vix,cpi_change,fedfunds_change,unrate_change,mrkt_expect_change,volatility_change
date,,,,,,,,,,,,
1927-12-31,17.660000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1928-01-31,17.570000,0.004574,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1928-02-29,17.260000,0.005828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1928-03-31,19.280001,0.017414,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1928-04-30,19.750000,0.003557,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31,6845.500000,-0.007358,326.031,3.72,4.4,4.18,14.95,NaN,NaN,NaN,0.04,0.043266
2026-01-31,6939.029785,-0.004302,326.588,3.64,4.3,4.26,17.44,NaN,NaN,NaN,0.02,0.033175
2026-02-28,6878.879883,-0.004339,327.460,3.64,4.4,3.97,19.86,NaN,NaN,NaN,-0.05,0.066023


### Fred API data

### yfinance data

In [ ]:
# stock data from yfinance
sp500 = yf.download("^GSPC", period="max")
sp500.head()
sp500.tail()
sp500.shape
sp500.index.min()
sp500.index.max()

---
## EDA

In [ ]:
print(sp500_macro.shape)
sp500_macro.dtypes
print(sp500_macro.head())
print(sp500_macro.isnull().sum().sort_values(ascending=False))
sp500_macro.duplicated().sum()
sp500_macro.describe()

(1181, 12)
                sp500    return  cpi  fedfunds  unrate  dgs10  vix  \
date                                                                 
1927-12-31  17.660000       NaN  NaN       NaN     NaN    NaN  NaN   
1928-01-31  17.570000  0.004574  NaN       NaN     NaN    NaN  NaN   
1928-02-29  17.260000  0.005828  NaN       NaN     NaN    NaN  NaN   
1928-03-31  19.280001  0.017414  NaN       NaN     NaN    NaN  NaN   
1928-04-30  19.750000  0.003557  NaN       NaN     NaN    NaN  NaN   

            cpi_change  fedfunds_change  unrate_change  mrkt_expect_change  \
date                                                                         
1927-12-31         NaN              NaN            NaN                 NaN   
1928-01-31         NaN              NaN            NaN                 NaN   
1928-02-29         NaN              NaN            NaN                 NaN   
1928-03-31         NaN              NaN            NaN                 NaN   
1928-04-30         NaN        

,sp500,return,cpi,fedfunds,unrate,dgs10,vix,cpi_change,fedfunds_change,unrate_change,mrkt_expect_change,volatility_change
count,1181.000000,1180.000000,950.000000,861.000000,938.000000,772.000000,436.000000,180.000000,90.000000,168.000000,772.000000,436.000000
mean,736.270405,0.000850,124.493271,4.600441,5.662260,5.810829,19.500619,0.001874,0.015000,0.014286,-0.009585,0.006669
std,1271.347311,0.010482,89.807806,3.537184,1.704982,2.951058,7.389689,0.004219,0.318519,0.315334,0.058826,0.060413
min,4.430000,-0.074534,21.480000,0.050000,2.500000,0.550000,9.510000,-0.009197,-1.050000,-1.500000,-0.430000,-0.206866
25%,24.830000,-0.003884,32.857500,1.910000,4.300000,3.887500,13.837500,-0.000352,-0.115000,-0.200000,-0.040000,-0.029531
50%,103.440002,0.000936,109.800000,4.290000,5.500000,5.430000,17.555000,0.001269,0.015000,0.000000,0.000000,0.002476
75%,1095.630005,0.005909,201.150000,6.120000,6.700000,7.532500,23.385000,0.002901,0.157500,0.200000,0.020000,0.037274
max,7126.060059,0.054175,330.293000,19.100000,14.800000,15.840000,59.890000,0.019643,1.090000,1.300000,0.240000,0.271568


---
## Data Cleaning

In [ ]:
print(sp500_macro.isna().sum())
print(sp500_macro.shape)
print(macro_df.isna().sum())
print(cpi.isna().sum())
print(cpi.shape)
print(interest.shape)
print(unemployment.shape)


sp500                    0
return                   1
cpi                    231
fedfunds               320
unrate                 243
dgs10                  409
vix                    745
cpi_change            1001
fedfunds_change       1091
unrate_change         1013
mrkt_expect_change     409
volatility_change      745
dtype: int64
(1181, 12)
date                      0
cpi                   16225
fedfunds              16314
unrate                16237
dgs10                  1118
vix                    8009
cpi_change            16995
fedfunds_change       17085
unrate_change         17007
mrkt_expect_change     2020
volatility_change      8414
dtype: int64
cpi    1
dtype: int64
(951, 1)
(861, 1)
(939, 1)
